# Unimodal Classification - Images
This notebook implements unimodal image classification for crisis-related data using deep learning. It covers data loading, preprocessing, model building, training, fine-tuning, and evaluation.


## Imports and Setup
Import all necessary libraries for data handling, visualization, model building, and evaluation. Set up the environment and load configuration from `.env` files.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import seaborn as sns
import tensorflow as tf

from itertools import cycle
from keras.callbacks import EarlyStopping, ModelCheckpoint
from keras.optimizers import AdamW
from sklearn.calibration import calibration_curve
from sklearn.metrics import (auc, classification_report, confusion_matrix, roc_auc_score, roc_curve)
from sklearn.preprocessing import LabelEncoder, label_binarize
from tensorflow import keras
from tqdm import tqdm

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# Load paths from .env file
dataset_dir = os.getenv("DATASET_DIR")
datasplits_dir = os.getenv("DATASPLITS_DIR")
models_dir = os.getenv("MODELS_DIR")

# Patgs to the data splits files
train_file = os.path.join(datasplits_dir, "img_train.tsv")
val_file = os.path.join(datasplits_dir, "img_val.tsv")
test_file = os.path.join(datasplits_dir, "img_test.tsv")

# Load the data splits into pandas DataFrames
train_df = pd.read_csv(train_file, sep="\t")
val_df = pd.read_csv(val_file, sep="\t")
test_df = pd.read_csv(test_file, sep="\t")

In [ ]:
train_df.shape

In [ ]:
train_df.head(1)

In [ ]:
val_df.shape

In [ ]:
val_df.head(1)

In [ ]:
test_df.shape

In [ ]:
test_df.head(1)

## Labels Encoding
Encode categorical labels for both informativeness and humanitarian categories using `LabelEncoder` to prepare them for model training.


In [ ]:
# Encode categorical labels for both image_info and image_human columns.
label_encoders = {}
for col in ["image_info", "image_human"]:
    le = LabelEncoder()
    # Concatenate all values from train, val, and test splits for this column
    all_values = pd.concat([train_df[col], val_df[col], test_df[col]])
    le.fit(all_values)
    # Transform each split in-place
    train_df[col] = le.transform(train_df[col])
    val_df[col] = le.transform(val_df[col])
    test_df[col] = le.transform(test_df[col])
    # Store the encoder for later inverse-transform or mapping
    label_encoders[col] = le

## Helper Functions
Define utility functions for:
- Loading and preprocessing images
- Creating TensorFlow datasets from DataFrames
- Building the model architecture
- Plotting training history and evaluation metrics

In [ ]:
def get_backbone(name, input_shape):
    """
    Returns a pretrained CNN backbone model (without top classification layers) and its preprocessing function.

    Args:
        name (str): Name of the backbone architecture. Supported: "ResNet50", "VGG16", "EfficientNetB0".
        input_shape (tuple): Shape of the input images, e.g., (224, 224, 3).

    Returns:
        model (tf.keras.Model): The base CNN model with ImageNet weights, no top.
        preprocess_fn (callable): The preprocessing function for the chosen model.

    Raises:
        ValueError: If an unknown model name is provided.
    """
    if name == "ResNet50":
        # Return ResNet50 backbone and its preprocessing function
        return keras.applications.ResNet50(input_shape=input_shape, include_top=False, weights="imagenet"), keras.applications.resnet50.preprocess_input
    elif name == "VGG16":
        # Return VGG16 backbone and its preprocessing function
        return keras.applications.VGG16(input_shape=input_shape, include_top=False, weights="imagenet"), keras.applications.vgg16.preprocess_input
    elif name == "EfficientNetB0":
        # Return EfficientNetB0 backbone and its preprocessing function
        return keras.applications.EfficientNetB0(input_shape=input_shape, include_top=False, weights="imagenet"), keras.applications.efficientnet.preprocess_input
    else:
        # Raise error if model name is not recognized
        raise ValueError(f"Unknown model name: {name}")

In [ ]:
def load_image_factory(preprocess_fn, image_size):
    """
    Factory function that creates an image loading and preprocessing function for tf.data pipelines.

    Args:
        preprocess_fn (callable): Preprocessing function specific to the chosen CNN backbone (e.g., ResNet50, VGG16).
        image_size (tuple): Target size for resizing images, e.g., (224, 224).

    Returns:
        load_image (callable): Function that takes an image path and labels, loads and preprocesses the image,
                               and returns (image_tensor, label_dict).
    """
    def load_image(path, label1, label2):
        # Read the image file from disk
        img = tf.io.read_file(path)
        # Decode the JPEG image to a tensor
        img = tf.image.decode_jpeg(img, channels=3)
        # Resize the image to the target size
        img = tf.image.resize(img, image_size)
        # Apply model-specific preprocessing (e.g., normalization)
        img = preprocess_fn(img)
        # Return the image tensor and a dictionary of labels
        return img, {"info": label1, "human": label2}
    return load_image

In [ ]:
def df_to_dataset(df, load_image_fn, batch_size=32, shuffle=True):
    """
    Converts a DataFrame of image paths and labels into a TensorFlow Dataset.

    Args:
        df (pd.DataFrame): DataFrame containing image paths and label columns.
        load_image_fn (callable): Function to load and preprocess images and labels.
        batch_size (int, optional): Number of samples per batch. Default is 32.
        shuffle (bool, optional): Whether to shuffle the dataset. Default is True.

    Returns:
        tf.data.Dataset: A batched and prefetched tf.data.Dataset ready for training or evaluation.
    """
    # Construct full image paths
    full_paths = df["image_path"].apply(lambda p: os.path.normpath(os.path.join(dataset_dir, p))).values
    # Prepare labels for both tasks
    labels1 = df["image_info"].values.astype("float32")  # For sigmoid binary output
    labels2 = df["image_human"].values                   # For multiclass output
    # Create a tf.data.Dataset from image paths and labels
    image_ds = tf.data.Dataset.from_tensor_slices((full_paths, labels1, labels2))
    # Map the image loading and preprocessing function
    image_ds = image_ds.map(load_image_fn, num_parallel_calls=tf.data.AUTOTUNE)
    # Shuffle if required (buffer size 700 is arbitrary but reasonable for moderate datasets)
    if shuffle:
        image_ds = image_ds.shuffle(700)
    # Batch and prefetch for performance
    return image_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

In [ ]:
def build_full_model(base_model, image_size, num_human_classes, dropout_rate=0.6, classifier_units=128):
    """
    Builds a full Keras model for multitask image classification with data augmentation.

    Args:
        base_model (tf.keras.Model): Pretrained CNN backbone (e.g., ResNet50, VGG16, EfficientNetB0) without top layers.
        image_size (tuple): Input image size, e.g., (224, 224).
        num_human_classes (int): Number of classes for the humanitarian task (multiclass output).
        dropout_rate (float, optional): Dropout rate for regularization. Default is 0.6.
        classifier_units (int, optional): Number of units in the dense classifier layer. Default is 128.

    Returns:
        tf.keras.Model: A compiled Keras model with two output heads:
            - "info": binary (sigmoid) for informativeness
            - "human": multiclass (softmax) for humanitarian category
    """
    # Input layer for images
    inputs = keras.Input(shape=(*image_size, 3))

    # Data augmentation pipeline (applied only during training)
    data_augmentation = keras.Sequential([
        keras.layers.RandomFlip("horizontal"),
        keras.layers.RandomRotation(0.1),
        keras.layers.RandomZoom(0.2),
    ])

    # Apply data augmentation
    x = data_augmentation(inputs)
    
	# Pass through the pretrained base model
    x = base_model(x)
    
	# Custom MLP
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Dense(classifier_units)(x)
    x = keras.layers.Activation("relu")(x)
    x = keras.layers.Dropout(dropout_rate)(x)

    # Output head for informativeness (binary classification)
    info_out = keras.layers.Dense(1, activation="sigmoid", name="info")(x)
    # Output head for humanitarian category (multiclass classification)
    human_out = keras.layers.Dense(num_human_classes, activation="softmax", name="human")(x)

    # Build and return the model
    return keras.Model(inputs=inputs, outputs=[info_out, human_out])

In [ ]:
def plot_training_history(history):
    """
    Plots training and validation metrics for each key in the Keras History object.

    Args:
        history (keras.callbacks.History): History object returned by model.fit().
    
    This function will create a separate plot for each metric (except those starting with 'val_'),
    showing both the training and validation curves for easy comparison.
    """
    for key in history.history:
        # Only plot metrics that are not validation metrics (those will be plotted together)
        if not key.startswith("val_"):
            plt.figure()
            plt.plot(history.history[key], label="train")
            plt.plot(history.history[f"val_{key}"], label="val")
            plt.title(key)
            plt.xlabel("Epoch")
            plt.ylabel("Value")
            plt.legend()
            plt.grid(True)
            plt.show()

In [ ]:
def partial_unfreeze(base_model, model, model_name):
    """
    Partially unfreezes the base model depending on architecture type.
    Applies custom unfreezing logic for ResNet50, VGG16, and EfficientNetB0.

    Args:
        base_model (keras.Model): The backbone model (without classification head).
        model (keras.Model): The full model (base + head).
        model_name (str): One of ["ResNet50", "VGG16", "EfficientNetB0"].

    Returns:
        None (modifies model in-place).
    """
    base_model.trainable = True  # Enable global setting; we'll fine-tune layer-by-layer

    set_trainable = False

    for layer in base_model.layers:

        # EfficientNetB0: Unfreeze from 'block5a' onward, skip BatchNorms
        if model_name == "EfficientNetB0":
            if "block5a" in layer.name:
                set_trainable = True

            if set_trainable and not isinstance(layer, keras.layers.BatchNormalization):
                layer.trainable = True
            else:
                layer.trainable = False

        # ResNet50: Unfreeze only conv5_block*, skip BatchNorms
        elif model_name == "ResNet50":
            if layer.name.startswith("conv5_block") and not isinstance(layer, keras.layers.BatchNormalization):
                layer.trainable = True
            else:
                layer.trainable = False

        # VGG16: Unfreeze block5 layers
        elif model_name == "VGG16":
            if layer.name.startswith("block5"):
                layer.trainable = True
            else:
                layer.trainable = False

        else:
            raise ValueError(f"Unknown model name: {model_name}")

    # Summary log
    print("Unfrozen layers:")
    for layer in base_model.layers:
        if layer.trainable:
            print(f"  {layer.name} ({layer.__class__.__name__})")

    print(f"\nTrainable {model_name.upper()} layers: {sum(layer.trainable for layer in base_model.layers)}")
    print(f"Trainable {model_name.upper()} weights: {len(base_model.trainable_weights)}")
    print(f"Total trainable weights in full model: {len(model.trainable_weights)}\n")


In [ ]:
def plot_reliability_curve(y_true, y_prob, task_name):
    """
    Plots a reliability diagram (calibration curve) for predicted probabilities.

    Args:
        y_true (array-like): True binary or multiclass labels.
        y_prob (array-like): Predicted probabilities for the positive class or each class.
        task_name (str): Name of the task for plot title.
    """
    # Compute calibration curve (fraction of positives vs. mean predicted value)
    prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=10)
    plt.figure(figsize=(6, 5))
    plt.plot(prob_pred, prob_true, marker='o', label='Model')
    plt.plot([0, 1], [0, 1], linestyle='--', label='Perfectly calibrated')
    plt.title(f"{task_name} - Reliability Diagram")
    plt.xlabel("Predicted probability")
    plt.ylabel("True probability")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

def plot_roc_auc(y_true, y_scores, class_names, task_name):
	"""
	Plots ROC curve(s) and computes AUC for binary or multiclass classification.

	Args:
		y_true (array-like): True labels.
		y_scores (array-like): Predicted probabilities or scores.
		class_names (list): List of class names for multiclass.
		task_name (str): Name of the task for plot title.
	"""
	if len(np.unique(y_true)) == 2:
		# Binary classification ROC curve
		roc_auc = roc_auc_score(y_true, y_scores)
		fpr, tpr, _ = roc_curve(y_true, y_scores)

		plt.figure(figsize=(6, 5))
		plt.plot(fpr, tpr, label=f"ROC curve (AUC = {roc_auc:.2f})")
		plt.plot([0, 1], [0, 1], "k--")
		plt.xlabel("False Positive Rate")
		plt.ylabel("True Positive Rate")
		plt.title(f"{task_name} - ROC Curve")
		plt.legend(loc="lower right")
		plt.grid(True)
		plt.tight_layout()
		plt.show()

	else:
		# Multiclass ROC curve (one-vs-rest for each class)
		n_classes = len(class_names)
		y_true_bin = label_binarize(y_true, classes=list(range(n_classes)))

		fpr = {}
		tpr = {}
		roc_auc = {}

		for i in range(n_classes):
			fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_scores[:, i])
			roc_auc[i] = auc(fpr[i], tpr[i])

		colors = cycle(["aqua", "darkorange", "cornflowerblue", "green", "red"])
		plt.figure(figsize=(8, 6))

		for i, color in zip(range(n_classes), colors):
			plt.plot(fpr[i], tpr[i], color=color, lw=2,
					 label=f"Class {class_names[i]} (AUC = {roc_auc[i]:.2f})")
		
		plt.plot([0, 1], [0, 1], "k--", lw=2)
		plt.xlim([0.0, 1.0])
		plt.ylim([0.0, 1.05])
		plt.xlabel("False Positive Rate")
		plt.ylabel("True Positive Rate")
		plt.title(f"{task_name} - ROC Curve")
		plt.legend(loc="lower right")
		plt.grid(True)
		plt.tight_layout()
		plt.show()


def evaluate_task(y_true, y_pred, task_name, class_names=None, y_scores=None):
    """
    Evaluates classification performance for a given task, printing a classification report,
    plotting a confusion matrix, and (optionally) ROC-AUC and reliability diagrams.

    Args:
        y_true (array-like): True labels.
        y_pred (array-like): Predicted labels.
        task_name (str): Name of the task (for plot/report titles).
        class_names (list, optional): List of class names for display.
        y_scores (array-like, optional): Predicted probabilities or scores for ROC/reliability plots.
    """
    # Print the classification report with precision, recall, f1-score, and support
    print(f"\n{task_name} - Classification Report:")
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

    # Prepare display names for confusion matrix (capitalize for Task #2)
    if task_name == "Task #2":
        display_names = [c[0].upper() for c in class_names]
    else:
        display_names = class_names

    # Compute and plot the confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap="Blues",
                xticklabels=display_names, yticklabels=display_names)
    plt.title(f"{task_name} - Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.show()
 
    # If probability scores are provided, plot ROC-AUC and reliability diagrams
    if y_scores is not None:
        plot_roc_auc(y_true, y_scores, class_names, task_name)
        # Only plot reliability diagram for binary tasks
        if len(np.unique(y_true)) == 2:
            plot_reliability_curve(y_true, y_scores, task_name)


def evaluate_model(model, test_ds):
    """
    Runs inference on the test dataset and evaluates the model on both tasks.

    Args:
        model (tf.keras.Model): Trained multitask model.
        test_ds (tf.data.Dataset): Test dataset.

    This function collects predictions and true labels for both tasks:
      - Task 1: informativeness (binary classification)
      - Task 2: humanitarian category (multiclass classification)
    It then calls evaluate_task() for each task to print metrics and plots.
    """
    # Initialize lists to collect true labels, predicted labels, and predicted probabilities for both tasks
    y_true_info, y_pred_info, y_scores_info = [], [], []
    y_true_human, y_pred_human, y_scores_human = [], [], []
    
    # Iterate over the test dataset in batches
    for img_batch, label_batch in tqdm(test_ds, desc="Running inference on test set"):
        # Predict outputs for each task
        preds = model.predict(img_batch, verbose=0)

        # Task 1: informativeness (binary)
        info_probs = preds[0].flatten()  # Predicted probabilities for binary task
        y_scores_info.extend(info_probs)
        y_true_info.extend(label_batch["info"].numpy())
        y_pred_info.extend((preds[0] > 0.5).astype("int32").flatten())  # Threshold at 0.5

        # Task 2: humanitarian category (multiclass)
        human_probs = preds[1]  # Predicted probabilities for each class
        y_scores_human.extend(human_probs)
        y_true_human.extend(label_batch["human"].numpy())
        y_pred_human.extend(np.argmax(preds[1], axis=1))  # Predicted class index

    # Evaluate Task 1: informativeness
    evaluate_task(
        y_true_info,
        y_pred_info,
        "Task #1",
        class_names=["Informative", "Not Informative"],
        y_scores=np.array(y_scores_info)
    )

    # Evaluate Task 2: humanitarian category
    evaluate_task(
        y_true_human,
        y_pred_human,
        "Task #2",
        class_names=[
            "affected_individuals",
            "infrastructure_and_utility_damage",
            "not_humanitarian",
            "other_relevant_information",
            "rescue_volunteering_or_donation_effort"
        ],
        y_scores=np.array(y_scores_human)
    )

## Hyperparameters
Define and document all key hyperparameters used for training and fine-tuning, such as image size, batch size, learning rates, and early stopping criteria.

In [ ]:
# Define image size for model input
IMAGE_SIZE = (224, 224)

# Batch size for training and evaluation
BATCH_SIZE = 32

# Training hyperparameters for initial (frozen) training phase
LR_TRAIN = 1e-4         # Learning rate
WD_TRAIN = 1e-5         # Weight decay (L2 regularization)
EPOCHS_TRAIN = 100      # Maximum number of epochs
PATIENCE_TRAIN = 5      # Early stopping patience

# Fine-tuning hyperparameters (after unfreezing backbone)
LR_FT = 1e-6            # Lower learning rate for fine-tuning
WD_FT = 1e-5            # Weight decay for fine-tuning
EPOCHS_FT = 20          # Maximum epochs for fine-tuning
PATIENCE_FT = 2         # Early stopping patience for fine-tuning

## Multimodel Evaluation Pipeline

This section runs the full training and evaluation pipeline for each selected CNN backbone (ResNet50, VGG16, EfficientNetB0). For each backbone, the following steps are performed:

1. **Load Base Model and Preprocessing**  
   The chosen backbone is loaded with pretrained ImageNet weights (excluding the top classification layers), and the appropriate preprocessing function is selected.

2. **Prepare Data Loaders**  
   TensorFlow datasets are created for training, validation, and testing splits, applying the backbone-specific preprocessing to all images.

3. **Build Model**  
   The multitask model is constructed by attaching custom classification heads to the backbone for both informativeness (binary) and humanitarian (multiclass) tasks.

4. **Compile Model**  
   The model is compiled with appropriate loss functions and metrics for each task, and an optimizer with weight decay is set.

5. **Train (Frozen Backbone)**  
   The model is trained with the backbone layers frozen, using early stopping and model checkpointing to prevent overfitting and save the best weights.

6. **Fine-Tune (Partial Unfreezing)**  
   The top layers of the backbone are unfrozen for fine-tuning, allowing the model to adapt pretrained features to the crisis dataset. Training continues with a lower learning rate.

7. **Evaluate Model**  
   The best fine-tuned model is loaded and evaluated on the test set. Performance metrics, confusion matrices, ROC curves, and reliability diagrams are generated for both tasks.

8. **Save Model**  
   The final trained model is saved for future inference or deployment.

This pipeline enables a fair comparison of different backbone architectures on the same crisis image classification tasks.

In [ ]:
backbones = ["ResNet50", "VGG16", "EfficientNetB0"]

for model_name in backbones:
    print(f"\nStarting pipeline for: {model_name.upper()}\n\n")

    # 1. Load base model + preprocessing
    base_model, preprocess_fn = get_backbone(model_name, input_shape=(*IMAGE_SIZE, 3))

    # 2. Prepare data loaders with preprocess_fn
    load_image = load_image_factory(preprocess_fn, IMAGE_SIZE)
    train_ds = df_to_dataset(train_df, load_image, BATCH_SIZE, shuffle=True)
    val_ds = df_to_dataset(val_df, load_image, BATCH_SIZE)
    test_ds = df_to_dataset(test_df, load_image, BATCH_SIZE, shuffle=False)

    # 3. Build model
    base_model.trainable = False
    model = build_full_model(base_model, IMAGE_SIZE, num_human_classes=train_df["image_human"].nunique())
    model.summary()

    # 4. Compile
    model.compile(
        optimizer=AdamW(LR_TRAIN, weight_decay=WD_TRAIN),
        loss={"info": "binary_crossentropy", "human": "sparse_categorical_crossentropy"},
        metrics={"info": "accuracy", "human": "accuracy"}
    )

    # 5. Train (frozen)
    print(f"\nTraining {model_name.upper()} model...")
    callbacks = [
        ModelCheckpoint(f"{models_dir}/{model_name}_trained.weights.h5", save_weights_only=True, save_best_only=True, monitor="val_loss", verbose=0),
        EarlyStopping(patience=PATIENCE_TRAIN, restore_best_weights=True)
    ]
    history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_TRAIN, callbacks=callbacks)
    plot_training_history(history)

    # 6. Fine-tune
    print(f"\nFine-tuning {model_name.upper()} model...")
    partial_unfreeze(base_model, model, model_name)
    model.compile(
        optimizer=AdamW(LR_FT, weight_decay=WD_FT),
        loss={"info": "binary_crossentropy", "human": "sparse_categorical_crossentropy"},
        metrics={"info": "accuracy", "human": "accuracy"}
    )
    callbacks = [
        ModelCheckpoint(f"{models_dir}/{model_name}_fine_tuned.weights.h5", save_weights_only=True, save_best_only=True, monitor="val_loss", verbose=0),
        EarlyStopping(patience=PATIENCE_FT, restore_best_weights=True)
    ]
    ft_history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FT, callbacks=callbacks)
    plot_training_history(ft_history)

    # 7. Evaluate
    print(f"\nEvaluating {model_name.upper()} model...")
    model.load_weights(f"{models_dir}/{model_name}_fine_tuned.weights.h5")
    evaluate_model(model, test_ds)

    # 8. Save Model
    model.save(f"{models_dir}/{model_name}_fine_tuned.model.keras")